# Offline D4RL AntMaze-UMaze-v2 — strict offline contrastive RL, 50k learner updates

Faithful **offline** contrastive RL (Eysenbach 2022 recipe, offline arm) on the
official `antmaze-umaze-v2` dataset. **No online collection anywhere.** No causal
changes yet — this is the offline baseline.

**What the run does**
- loads a *fixed* dataset once, freezes the replay buffer, and trains **only** by
  sampling that buffer — there is no training environment, no collector, no random
  warm-up, and no training-time `env.step()`;
- budget is counted in **learner updates** (gradient steps), stopping at **50k**;
- gates at `[1000, 5000, 10000, 25000, 50000]` learner updates, each with a full
  diagnostic report (JSON + Markdown) and hard STOP conditions.

**Run identity (single source of truth below)**

> `RUN_ID = offline_umaze_bc005_twinmin_s0_50k` is a **run / output directory
> name**, *not* a dataset name. It encodes the recipe: bc_coef **0.05**, twin-Q
> with actor **min(Q1,Q2)**, seed **0**, **50k** updates. The dataset it trains
> on is the separate NPZ at `DATASET_DRIVE_PATH`.

Open in Colab and run top-to-bottom. It **stops at 50k** and does not auto-continue.


## 1. Configuration — single source of truth

In [ ]:
# 1. Configuration. Everything tunable lives here; nothing below needs editing.
import os

SEED   = 0
MAX_UPDATES = 50_000                     # hard stop (learner updates); no auto-continue
GATES  = [1000, 5000, 10000, 25000, 50000]
RESUME = False                           # set True after a Colab disconnect
REQUIRE_GPU = True

# --- dataset (a fixed NPZ; NOT produced by this run) -------------------------
DATASET_DRIVE_PATH = "/content/drive/MyDrive/contrastive_rl_datasets/d4rl/antmaze_umaze_v2_offline.npz"
LOCAL_DATASET_PATH = "/content/antmaze_umaze_v2_offline.npz"

# --- run / output directory (NOT a dataset) ---------------------------------
# RUN_ID names the OUTPUT dir; it encodes the recipe (bc0.05, twin-min, s0, 50k).
RUN_ID        = "offline_umaze_bc005_twinmin_s0_50k"
RUN_DRIVE_DIR = "/content/drive/MyDrive/contrastive_rl_runs/offline_antmaze/offline_umaze_bc005_twinmin_s0_50k"
LOCAL_RUN_DIR = f"/content/scratch_{RUN_ID}"     # fast local scratch (atomic writes)

# --- official dataset provenance (for the optional download+convert cell) ----
D4RL_HDF5_URL  = "http://rail.eecs.berkeley.edu/datasets/offline_rl/ant_maze_v2/Ant_maze_u-maze_noisy_multistart_False_multigoal_False_sparse_fixed.hdf5"
D4RL_HDF5_SHA_PREFIX = "5ef15257"          # audited official file (soft provenance check)
D4RL_HDF5_SIZE = 232_532_949               # bytes
LOCAL_HDF5     = "/content/antmaze-umaze-v2.hdf5"
HDF5_DRIVE_PATH = "/content/drive/MyDrive/contrastive_rl_datasets/d4rl/antmaze-umaze-v2.hdf5"

# --- repo -------------------------------------------------------------------
REPO_URL = "https://github.com/tingrui-huang/contrastive_rl.git"
BRANCH   = "main"
COMMIT   = ""                            # ""=track origin/BRANCH; pin a hash when archiving
REPO     = "/content/contrastive_rl"

print("RUN_ID        :", RUN_ID, "(this is an OUTPUT directory, not a dataset)")
print("RUN_DRIVE_DIR :", RUN_DRIVE_DIR)
print("dataset (Drive):", DATASET_DRIVE_PATH)
print("dataset (local):", LOCAL_DATASET_PATH)
print("budget: %d learner updates | gates %s | RESUME=%s" % (MAX_UPDATES, GATES, RESUME))

## 2. Mount Google Drive

In [ ]:
# 2. Mount Drive; create the local scratch + Drive run trees.
from google.colab import drive
drive.mount("/content/drive")
for base in (LOCAL_RUN_DIR, RUN_DRIVE_DIR):
    for d in ("checkpoints", "gates", "reports", "logs"):
        os.makedirs(f"{base}/{d}", exist_ok=True)
os.makedirs(os.path.dirname(DATASET_DRIVE_PATH), exist_ok=True)
print("local scratch:", LOCAL_RUN_DIR)
print("Drive run dir:", RUN_DRIVE_DIR)
!df -h /content | tail -1

## 3. Clone/checkout the repo (never overwrites uncommitted work)

In [ ]:
# 3. Clone if absent; otherwise fetch + checkout safely.
import subprocess, sys
os.chdir("/content")
if not os.path.exists(REPO):
    !git clone $REPO_URL $REPO
os.chdir(REPO)
dirty = subprocess.run(["git","status","--porcelain"], capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError("repo has uncommitted changes — refusing to checkout over them:\n" + dirty)
!git fetch -q origin
ref = COMMIT if COMMIT else f"origin/{BRANCH}"
!git checkout -q $ref
!git log -1 --oneline
for req in ("crl/offline_audit.py","scripts/audit_offline.py","crl/d4rl_ant.py",
            "scripts/verify_offline_d4rl.py","scripts/offline_qual_50k.py",
            "scripts/offline_gate_report.py","scripts/convert_d4rl_to_npz.py"):
    if not os.path.exists(req):
        raise RuntimeError(f"{req} missing from the checkout — push main from the "
                           "workstation (git push origin main) and rerun this cell.")
print("offline pipeline files present — checkout OK")

## 4. Dependencies (proven-safe Colab recipe — do not alter)

Verbatim from `antmaze_umaze_qualification.ipynb`: install the jax-adjacent libs
with `--no-deps` and **hold** Colab's GPU-matched `jax/jaxlib/numpy`, or the kernel
hard-crashes on the next GPU op.

In [ ]:
# 4. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
import jax, jaxlib, numpy
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}", f"numpy=={numpy.__version__}"]
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps", "dm-haiku", "optax", "chex")
pip("jmp", "tabulate", "toolz", "etils", "tensorboardX",
    "gymnasium-robotics", "mujoco", "imageio-ffmpeg", "h5py", *hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())

## 5. Environment / GPU verification

In [ ]:
# 5. Versions + accelerator check -> Drive.
import hashlib, json, platform, importlib.metadata as im
import mujoco
os.chdir(REPO)
commit = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
env_meta = {"run_id": RUN_ID, "python": platform.python_version(),
            "jax": jax.__version__, "jaxlib": jaxlib.__version__,
            "backend": jax.default_backend(), "devices": [str(d) for d in jax.devices()],
            "mujoco": mujoco.__version__, "gymnasium": im.version("gymnasium"),
            "git_commit": commit}
if REQUIRE_GPU and jax.default_backend() == "cpu":
    raise RuntimeError("no accelerator — Runtime > Change runtime type > GPU")
!nvidia-smi -L || true
json.dump(env_meta, open(f"{RUN_DRIVE_DIR}/meta_env.json","w"), indent=2)
print(json.dumps(env_meta, indent=1))

## 6. Dataset handling (fixed NPZ; never falls back to online collection)

- If the Drive NPZ exists: record its size + **SHA-256**, copy it to `/content`,
  and **re-verify the copied SHA** (byte-identical).
- If it is **absent**: this cell **stops** with clear instructions. Run the
  optional **§6b** download+convert cell (default OFF) to build it from the
  audited official D4RL HDF5, then re-run this cell.
- There is **no** online-collection fallback anywhere in this notebook.

In [ ]:
# 6. Verify Drive NPZ -> copy to /content -> verify copied SHA. Stop if absent.
import shutil
def sha256(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

if not os.path.exists(DATASET_DRIVE_PATH):
    raise FileNotFoundError(
        "\n=== DATASET MISSING ON DRIVE ===\n"
        f"expected: {DATASET_DRIVE_PATH}\n"
        "This notebook NEVER collects data online. To create the NPZ, set\n"
        "DOWNLOAD_AND_CONVERT=True in the next cell (§6b) and run it once; it\n"
        "downloads the audited official D4RL HDF5 and converts it with the\n"
        "repo converter, saving the NPZ to the Drive path above. Then re-run\n"
        "THIS cell.")

drive_sha = sha256(DATASET_DRIVE_PATH)
drive_sz  = os.path.getsize(DATASET_DRIVE_PATH)
print(f"Drive NPZ: {drive_sz/1e6:.1f} MB  sha256={drive_sha[:16]}…")
shutil.copy2(DATASET_DRIVE_PATH, LOCAL_DATASET_PATH + ".tmp")
os.replace(LOCAL_DATASET_PATH + ".tmp", LOCAL_DATASET_PATH)
local_sha = sha256(LOCAL_DATASET_PATH)
assert local_sha == drive_sha, f"copied SHA mismatch: {local_sha} != {drive_sha}"
assert os.path.getsize(LOCAL_DATASET_PATH) == drive_sz, "copied size mismatch"
DATASET_SHA = local_sha
print(f"copied -> {LOCAL_DATASET_PATH}  sha VERIFIED ({local_sha[:16]}…)")

## 6b. (optional) Download + convert the official D4RL HDF5 → NPZ

**Default OFF.** Only needed the first time, when the NPZ is not yet on Drive.
Downloads the official `antmaze-umaze-v2` HDF5 (rail.eecs.berkeley.edu), verifies
its full SHA-256, converts it with the repo's converter
(`scripts/convert_d4rl_to_npz.py`, zero-padded 29-D goal contract), and saves the
NPZ to `DATASET_DRIVE_PATH`. Then re-run **§6**.

In [ ]:
# 6b. Optional one-time dataset builder. Set True to run.
DOWNLOAD_AND_CONVERT = False

if DOWNLOAD_AND_CONVERT:
    os.chdir(REPO)
    # 1) obtain the HDF5 (prefer a Drive copy if you already downloaded it once)
    if os.path.exists(HDF5_DRIVE_PATH):
        print("using HDF5 from Drive:", HDF5_DRIVE_PATH)
        shutil.copy2(HDF5_DRIVE_PATH, LOCAL_HDF5)
    else:
        print("downloading official D4RL HDF5 …")
        subprocess.run(["wget","-q","-O",LOCAL_HDF5,D4RL_HDF5_URL], check=True)
        os.makedirs(os.path.dirname(HDF5_DRIVE_PATH), exist_ok=True)
        shutil.copy2(LOCAL_HDF5, HDF5_DRIVE_PATH)   # cache for next time
    h5_sha = sha256(LOCAL_HDF5)
    h5_sz  = os.path.getsize(LOCAL_HDF5)
    print(f"HDF5 {h5_sz/1e6:.1f} MB  sha256={h5_sha}")
    if not h5_sha.startswith(D4RL_HDF5_SHA_PREFIX) or h5_sz != D4RL_HDF5_SIZE:
        raise RuntimeError(
            f"HDF5 provenance mismatch: expected size {D4RL_HDF5_SIZE} and sha "
            f"prefix {D4RL_HDF5_SHA_PREFIX}, got size {h5_sz} sha {h5_sha[:8]}. "
            "Refusing to convert an unexpected file.")
    print("HDF5 provenance OK (audited official antmaze-umaze-v2).")
    # 2) convert with the repo converter (env-driven paths), save NPZ to Drive
    os.environ["D4RL_HDF5"]  = LOCAL_HDF5
    os.environ["OFFLINE_NPZ"] = DATASET_DRIVE_PATH
    sys.path.insert(0, f"{REPO}/scripts")
    import importlib, convert_d4rl_to_npz
    importlib.reload(convert_d4rl_to_npz)
    meta = convert_d4rl_to_npz.convert(LOCAL_HDF5, DATASET_DRIVE_PATH)
    print("saved NPZ to Drive:", DATASET_DRIVE_PATH)
    print("Now re-run §6 to copy + SHA-verify it into /content.")
else:
    print("DOWNLOAD_AND_CONVERT is False — skipping (dataset expected on Drive).")

## 7. Pre-training offline audit — must PASS before any training

Runs the real dataset through both offline correctness tools. **Any failure
stops the notebook** (training never starts):
- `crl/offline_audit.py` — fingerprint + static/buffer gates (immutability,
  key separation, shapes, relabel bounds, frozen buffer);
- `scripts/audit_offline.py` — the strict-offline gate suite incl. a
  *structurally-impossible-collection* smoke;
- `scripts/verify_offline_d4rl.py` — the 7 faithful-offline pre-training gates
  (BC-tuple alignment, zero-padded goal, **twin-min in the actor**, BC gradient
  ascends log-prob, dry-run ingest, runtime asserts armed, run-config contract).

In [ ]:
# 7. Offline audits on the REAL dataset. Training does not start unless all pass.
os.chdir(REPO)
os.environ["OFFLINE_NPZ"] = LOCAL_DATASET_PATH     # point every offline tool at the local copy

# (a) crl/offline_audit fingerprint + static gates
sys.path.insert(0, REPO)
from crl import offline_audit
from crl.config import Config
from crl import envs as envs_mod
_c = Config(env_name="offline_ant_umaze", offline_dataset=LOCAL_DATASET_PATH)
envs_mod.make_env("offline_ant_umaze", _c, seed=0)   # fills dims
passed, gates, rep = offline_audit.run_static_audit(LOCAL_DATASET_PATH, _c)
print("crl/offline_audit gates:", {g: ("PASS" if ok else "FAIL") for g,ok in gates.items()})
fp = rep["fingerprint"]
print(f"  sha256={fp['sha256'][:16]}…  eps={fp['n_episodes']}  trans={fp['n_transitions']}  obs={fp['obs_shape']}")
assert passed, "crl/offline_audit FAILED — see gates above"

# (b) scripts/audit_offline.py (env_name required: the d4rl npz carries no meta env_name)
r = subprocess.run([sys.executable, "scripts/audit_offline.py",
                    "--dataset", LOCAL_DATASET_PATH, "--env_name", "offline_ant_umaze"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode != 0:
    print(r.stderr[-1500:]); raise RuntimeError("scripts/audit_offline.py FAILED")

# (c) scripts/verify_offline_d4rl.py — 7 faithful-offline pre-training gates
r = subprocess.run([sys.executable, "scripts/verify_offline_d4rl.py"],
                   capture_output=True, text=True, env={**os.environ})
print(r.stdout[-1800:])
if r.returncode != 0 or "ALL GATES PASS" not in r.stdout:
    print(r.stderr[-1500:]); raise RuntimeError("verify_offline_d4rl gates FAILED")
print("\nALL OFFLINE AUDITS PASSED — safe to train.")

## 8. The offline objective (what is actually optimized)

Training tuple (single episode, geometric future relabeling):

$$ (s_i,\; a_i,\; g = s_j),\quad j > i,\ \ j\sim\text{Geom}(1-\gamma),\ \gamma=0.99 $$

- **Critic**: binary NCE on **twin** logits `f(s,a,g)` (2 heads).
- **Actor** (offline, per-sample):
  $$ L_{\text{actor}} = 0.05\,\big(-\log\pi(a_i\mid s_i,g)\big)\;+\;0.95\,\big(\alpha\log\pi(a\mid s_i,g) - \min(Q_1,Q_2)(s_i,a,g)\big),\quad \alpha=0. $$
  The first term is **behavior cloning** of the dataset action `a_i`; the second
  uses **min(Q1,Q2)** (pessimistic), *not* the mean.
- **Evaluation goal**: `[g_x, g_y, 0, …, 0]` — the XY target zero-padded to 29-D.
- **Success**: XY distance to the goal `≤ 0.5`.

`random_goals = 0.0` (goals are same-episode future relabels only). No causal
changes.

## 9. TensorBoard

In [ ]:
# 9. TensorBoard on the local scratch checkpoint dir (mirrored to Drive per gate).
tb_logdir = f"{LOCAL_RUN_DIR}/checkpoints/tb"
%load_ext tensorboard
%tensorboard --logdir $tb_logdir

## 10. Launch staged offline training → 50k learner updates

Runs `scripts/offline_qual_50k.py`, the strict-offline staged driver:
- gates at nominal `[1000, 5000, 10000, 25000, 50000]` → nearest block multiples
  `[1400, 4900, 9800, 25200, 50400]` (the offline step-clock advances 700 gradient
  updates per block);
- at each gate: atomic **local** `latest.pkl`/`gate_*.pkl` write, full gate report
  (JSON + Markdown), then **mirror to Drive**;
- after the 10k gate: a **checkpoint/resume determinism test** (params, twin
  critics, optimizer states, learner+sampler RNG, learner counter, dataset SHA);
- hard STOP on: replay change, any training collection, relabel crossing episodes,
  non-finite loss, twin-min not used, or resume SHA mismatch.

Re-running this cell after a disconnect **restores from Drive and skips finished
gates**. It stops at 50400 and never continues to 1M.

In [ ]:
# 10. Launch. Env vars point the driver at the local scratch + Drive mirror.
# Captures the driver output and surfaces its STDERR + per-stage log on any
# failure, instead of a bare non-zero exit.
import glob, pickle
os.chdir(REPO)
os.environ["OFFLINE_NPZ"]       = LOCAL_DATASET_PATH   # fixed dataset (local copy)
os.environ["OFFLINE_RUN_DIR"]   = LOCAL_RUN_DIR        # atomic local writes
os.environ["OFFLINE_DRIVE_DIR"] = RUN_DRIVE_DIR        # persistent Drive mirror
os.environ["PYTHONUNBUFFERED"]  = "1"

r = subprocess.run([sys.executable, "-u", "scripts/offline_qual_50k.py"],
                   env={**os.environ}, capture_output=True, text=True)
print(r.stdout[-8000:])
if r.returncode != 0:
    print("=========== DRIVER STDERR (tail) ===========")
    print(r.stderr[-8000:])
    for lg in sorted(glob.glob(f"{LOCAL_RUN_DIR}/logs/*.log")):
        print(f"=========== {lg} (tail) ===========")
        print(open(lg, encoding="utf-8", errors="replace").read()[-4000:])
    raise RuntimeError("offline qualification driver exited non-zero — see STDERR + logs above.")
print("driver finished. Latest step:")
lp = f"{RUN_DRIVE_DIR}/checkpoints/latest.pkl"
if os.path.exists(lp):
    print("  ", pickle.load(open(lp, "rb"))["step"], "learner updates")


## 11. Gate reports (JSON + Markdown, on Drive)

In [ ]:
# 11. Show each gate's Markdown report + collect the JSON summary.
import glob
from IPython.display import Markdown, display
rows = []
for g in GATES:
    mdp = f"{RUN_DRIVE_DIR}/reports/gate_{g}.md"
    jsp = f"{RUN_DRIVE_DIR}/reports/gate_{g}.json"
    if os.path.exists(mdp):
        display(Markdown(open(mdp, encoding="utf-8").read()))
    if os.path.exists(jsp):
        R = json.load(open(jsp))
        rows.append((g, R["step"], R["evaluation"]["success"],
                     R["evaluation"]["min_xy_dist_mean"], R["stop_flags"]))
print("\ngate | step | success | min_xy_dist | stop_flags")
for g, s, su, dmin, st in rows:
    print(f"{g:>6} | {s:>6} | {su:.3f}   | {dmin:.3f}      | {st or 'none'}")

## 12. Resume after a disconnect

1. Reopen this notebook; if the runtime is wedged, `Runtime > Disconnect and delete runtime`.
2. Run §1–§5 (config, Drive, clone, deps, env-verify) and §6 (dataset copy+SHA).
3. Set `RESUME = True` in §1 (optional — the driver auto-resumes from the latest
   Drive checkpoint regardless), then run §7 (audit) and §10 (launch).
4. The driver restores the Drive run dir to local scratch, **re-runs the offline
   audit**, **rejects a dataset-SHA or config mismatch**, and continues from the
   last completed gate. It will not restart from 0.

In [ ]:
# 12. Inspect saved metrics WITHOUT touching training.
mp = f"{RUN_DRIVE_DIR}/checkpoints/metrics.json"
if os.path.exists(mp):
    for e in json.load(open(mp))[-6:]:
        print({k: e.get(k) for k in ("step","success","min_dist","final_dist",
                                     "actor_loss","critic_loss","bc_nll")})
else:
    print("no metrics yet — run §10 first.")